# v10 interpretability, part 1: H0 record · H6 is the self real? · H1 is it something known? (GPU 0, no training)
**One language: English → Hindi** (FLORES dev, 997 sentences, teacher forcing).

Models: Hindi-only phase 45 · continual distant phase 10 and 40 (both Hindi phases) · an **untrained model** (random weights) as the control.

- **H6:** size of the largest co-firing group as the threshold goes 0.05 → 0.5, for the real activity vs 3 nulls (randomly rotated neurons, shuffled tokens, a random graph with the same number of links).
- **H1:** per-neuron scores for known neuron families (entropy neurons, token-frequency neurons, massive activations, universal-neuron signs, position/token neurons, language-specific neurons); are self neurons over-represented?

Results: `analysis/v10/interp/` (report.txt + figures). Takes roughly 10–20 min per model.

In [ ]:
import os, sys, glob, json, time, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
GPU = "0"
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
from IPython.display import Image, display
from huggingface_hub import HfApi, hf_hub_download
for attempt in range(5):
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; FILES = api.list_repo_files(HF_REPO); break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)

def get(run, phases):
    for ph in phases:
        if glob.glob(f"runs/{run}/ckpt/p{ph:03d}_*.pt"):
            continue
        name = [f for f in FILES if f.startswith(f"{run}/ckpt/p{ph:03d}_")]
        if name:
            hf_hub_download(HF_REPO, name[0], local_dir="runs")
        else:
            print("missing (not here, not on HF):", run, ph)

def last_phase(run):
    phs = [int(os.path.basename(f)[1:4]) for f in FILES if f.startswith(f"{run}/ckpt/p")]
    phs += [int(os.path.basename(f)[1:4]) for f in glob.glob(f"runs/{run}/ckpt/p*.pt")]
    return max(phs)


def run(*cmd):
    env = {**os.environ, "CML_LANGSET": "distant"}
    p = subprocess.Popen([sys.executable, "-u", "interp_v10.py", "--gpu", GPU, *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        print(line, end="")
    p.wait(); print("exit code", p.returncode)

MODELS = [("hindi45", "v4b_only_hi_s0", 45), ("cont40", "v4b_continual_s0", 40), ("cont10", "v4b_continual_s0", 10)]
for _, r, ph in MODELS:
    get(r, [ph])
need = ["data/en-hi/train.jsonl", "tokenizer_v4/spm.model"] + [f"data/flores/dev.{l}.jsonl" for l in ["en", "hi", "zh", "ar"]]
miss = [f for f in need if not os.path.exists(f)]
print("all there" if not miss else f"MISSING: {miss}")

In [ ]:
# old results from an earlier call are appended to, so start clean
import shutil; shutil.rmtree("analysis/v10/interp", ignore_errors=True)
for n, r, ph in MODELS:
    run("--model", f"{n}={r}:{ph}")
run("--model", "random=random")

In [ ]:
subprocess.run([sys.executable, "interp_v10.py", "--summarize"])
for f in ["h6_giant_vs_tau.png", "h1_enrichment.png"]:
    display(Image(f"analysis/v10/interp/{f}"))